In [1]:
import pandas as pd

file_path = r'D:\Aa中工互联\工作安排\英格索兰\data\特征工程\1.时间特征.xlsx'
df = pd.read_excel(file_path)

In [2]:
df.columns

Index(['时间', 'DQ200系统压力', 'AVS系统压力', 'DQ200累积流量', 'AVS累积流量', 'DQ200差分值',
       'AVS差分值', '瞬时流量', 'year', 'month', 'day', 'weekday', 'hour', 'minute',
       'second', 'is_weekend', 'quarter'],
      dtype='object')

In [ ]:
# 导入必要的库
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [7]:

# 假设你要预测的是 '是否故障' 这一列
X = df.drop(columns=['瞬时流量','时间'])   # 所有其他列作为特征
y = df['瞬时流量']                  # 目标变量

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 仅用训练数据fit scaler
X_train_scaled = scaler.fit_transform(X_train)

# 对测试数据transform，不要重新fit
X_test_scaled = scaler.transform(X_test)

# 如果你想检查结果，可以将数组转回DataFrame
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# 查看前几行数据
print(X_train_scaled_df.head())

   DQ200系统压力   AVS系统压力  DQ200累积流量   AVS累积流量  DQ200差分值    AVS差分值  year  month  \
0   0.394643  0.370712  -1.430360 -1.018164 -0.591599  0.729848   0.0    0.0   
1   0.342202  0.385339  -1.430313 -1.018115  1.050465  1.426380   0.0    0.0   
2   0.335647  0.383249  -1.430313 -1.018115 -0.591599 -0.663216   0.0    0.0   
3   0.368423  0.362354  -1.430313 -1.018067 -0.591599  1.426380   0.0    0.0   
4   0.427418  0.351907  -1.430313 -1.018051 -0.591599  0.033316   0.0    0.0   

       day   weekday     hour    minute    second  is_weekend  quarter  
0 -1.57966 -0.251385 -1.62333 -1.703055 -1.303599   -0.745354      0.0  
1 -1.57966 -0.251385 -1.62333 -1.703055 -1.013914   -0.745354      0.0  
2 -1.57966 -0.251385 -1.62333 -1.703055 -0.724229   -0.745354      0.0  
3 -1.57966 -0.251385 -1.62333 -1.703055 -0.434544   -0.745354      0.0  
4 -1.57966 -0.251385 -1.62333 -1.703055 -0.144859   -0.745354      0.0  


In [9]:
# 创建XGBoost回归模型
xgb_regressor = xgb.XGBRegressor(
    objective='reg:squarederror',  # 回归任务
    n_estimators=100,             # 树的数量
    max_depth=3,                  # 树的最大深度
    learning_rate=0.1,            # 学习率
    subsample=0.8,                # 样本采样比例
    colsample_bytree=0.8,         # 特征采样比例
    random_state=42
)

# 训练模型
xgb_regressor.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [10]:
# 进行预测
y_pred = xgb_regressor.predict(X_test)

# 评估模型
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared: {r2:.4f}")

Mean Squared Error: 4.0061
R-squared: 0.9855


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")

Root Mean Squared Error (RMSE): 2.0015
Mean Absolute Error (MAE): 1.3001
Mean Absolute Percentage Error (MAPE): 3953180109708066.50%
Adjusted R-squared: 0.9855


In [12]:
# 获取训练集目标变量的均值和标准差
train_target_mean = y_train.mean()
train_target_std = y_train.std()

# 计算所有预测值的 Z-score
z_scores = (y_pred - train_target_mean) / train_target_std

# 统计落在不同区间内的比例
normal_mask = np.abs(z_scores) < 1
moderate_mask = (np.abs(z_scores) >= 1) & (np.abs(z_scores) < 2)
anomaly_mask = np.abs(z_scores) >= 2

normal_ratio = np.mean(normal_mask)
moderate_ratio = np.mean(moderate_mask)
anomaly_ratio = np.mean(anomaly_mask)

# 输出结果
print(f"预测值在 ±1σ 范围内的比例: {normal_ratio * 100:.2f}%")
print(f"预测值在 ±2σ 范围内的比例（但 >1σ）: {moderate_ratio * 100:.2f}%")
print(f"预测值超出 ±2σ 的比例: {anomaly_ratio * 100:.2f}%")

预测值在 ±1σ 范围内的比例: 71.10%
预测值在 ±2σ 范围内的比例（但 >1σ）: 28.88%
预测值超出 ±2σ 的比例: 0.01%


# 预测新数据

In [ ]:
# 预测新数据示例（假设有新数据）
new_data = np.array([[0.01, -0.04, 0.03, -0.03, 0.01, 0.02, -0.02, 0.05, 0.02, 0.01]])
predicted_value = xgb_regressor.predict(new_data)
print(f"\nPredicted value for new data: {predicted_value[0]:.4f}")